# Lenormand B15 — Qwen3.8-27B Latent Risk Readout

这不是再训练一个 27B，也不是生成 CoT。B15 检验一个很具体的假设：**模型在回答前的中间层已经把四级风险分开了，但最终 A/B token head 丢失了部分信息。**

对每篇帖子仍然只问原来四张 Risk Card；单次 forward 同时截取 decoder 第 16/32/48/64 层最后一个 token 的 hidden state。随后仅在 outer-train 上训练一个共享线性 probe，并在从未参与 probe/SFT 的 outer-valid 上与 `B4_FIXED_BLEND` 成对比较。

- Stage 1：Fold 0 screening，允许从四层和三个正则强度中选择一次。
- Stage 2：只有 Fold 0 主候选达到 `Δ weighted-F1 ≥ +0.008` 才运行；Fold 1/2 **冻结 Fold 0 的 layer、C 和 25% blend**。
- 可见 CoT 上次 `48/48` 被截断且 `parsed FINAL = 0`，属于无效生成实验；silent/pairwise rank 才是已确认失败。本 notebook 不烧生成式 CoT。
- 预计 A100 80GB：Fold 0 约 **45–70 分钟**；若通过，另外两折合计约 **1.5–2.3 小时**。所有 hidden chunks 均保存到 Drive，可断线恢复。


In [ ]:
#@title 0A. 新 runtime 安装基础依赖
%%capture
!pip install -q -U   "transformers>=5.8.0"   "accelerate>=1.6.0"   "peft>=0.17.0"   "bitsandbytes>=0.46.0"   "sentencepiece>=0.2.0"   "openpyxl>=3.1.0"   "scikit-learn>=1.6.0,<1.8.0"   "scipy>=1.13.0"   "kernels"


In [ ]:
#@title 0B. Qwen3.8 混合架构 kernels（安装后必须重启 session）
!pip install -U "flash-linear-attention[cuda]"
!pip install -U causal-conv1d --no-build-isolation

print('现在 Runtime → Restart session；重启后从第 1 格开始，不要重跑 0A/0B。')


In [ ]:
#@title 1. Drive、文件和运行开关
from google.colab import drive, files
drive.mount('/content/drive')

from pathlib import Path
import gc, importlib, json, shutil, subprocess, sys, time

ROOT = Path('/content/drive/MyDrive/IEEE_BigData2026')
TRAIN_PATH = ROOT / 'train.xlsx'
if not TRAIN_PATH.exists(): TRAIN_PATH = ROOT / 'ieee/train.xlsx'

FOLD_SOURCE = ROOT / 'results/B4P_AVC_FAST3/B4P_CORE_OOF.npz'
OUT = ROOT / 'results/B15_LATENT_RISK_READOUT'
OUT.mkdir(parents=True, exist_ok=True)

# 第一次只跑 Fold 0。若第 6 格显示 PASS，再把 RUN_CONFIRMATION 改 True。
RUN_FOLD0 = True
RUN_CONFIRMATION = False

MODULE_MARKERS = {
    'b1_experiments.py': None,
    'b4p_anchor_verifier.py': 'B4P_RUNTIME_REVISION = "2026-08-21.qwen38-full64-kernels-v4"',
    'b4_task1_q38.py': 'TASK1_RUNTIME_REVISION = "2026-08-24.q38-full64-official-evidence-v4"',
    'b8_risk_only.py': 'B8R_RUNTIME_REVISION',
    'b15_latent_readout.py': 'B15_RUNTIME_REVISION = "2026-08-30.latent-risk-readout-v1"',
}
stale = []
for name, marker in MODULE_MARKERS.items():
    path = ROOT / name
    if not path.exists() or (marker and marker not in path.read_text(encoding='utf-8')):
        stale.append(name)
if stale:
    print('上传并覆盖这些模块：', stale)
    uploaded = files.upload()
    for name in stale:
        if name not in uploaded: raise FileNotFoundError(name)
        shutil.copy2('/content/' + name, ROOT / name)

required = [TRAIN_PATH, FOLD_SOURCE]
for fold in range(3):
    candidates = [
        ROOT / f'results/B4_TASK1_Q38_FULL64_FOLD0/Q38_FULL64/fold_{fold}/frozen_task1_spec.json',
        ROOT / f'results/B4_TASK1_Q38_FULL64_OUTER_CONFIRM/Q38_FULL64/fold_{fold}/frozen_task1_spec.json',
    ]
    if not any(path.exists() for path in candidates):
        required.append(candidates[-1])
missing = [str(path) for path in required if not path.exists()]
if missing: raise FileNotFoundError('缺少冻结产物：\n' + '\n'.join(missing))

sys.path.insert(0, str(ROOT))
print(subprocess.run(
    ['nvidia-smi','--query-gpu=name,memory.total,driver_version','--format=csv,noheader'],
    capture_output=True, text=True,
).stdout)
print({'output': str(OUT), 'drive_free_gb': round(shutil.disk_usage(ROOT).free / 2**30, 2)})


In [ ]:
#@title 2. 环境、revision 与数据硬检查
import numpy as np
import pandas as pd
import torch
import transformers
import sklearn

import b1_experiments as b1
import b4p_anchor_verifier as b4
import b4_task1_q38 as task1
import b8_risk_only as b8
import b15_latent_readout as b15
importlib.reload(b1); importlib.reload(b4); importlib.reload(task1); importlib.reload(b8); importlib.reload(b15)

assert b15.B15_RUNTIME_REVISION == '2026-08-30.latent-risk-readout-v1'
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
kernel = b4.qwen35_kernel_status()
print({
    'transformers': transformers.__version__, 'torch': torch.__version__,
    'sklearn': sklearn.__version__, 'gpu_gb': gpu_gb, 'kernel': kernel,
})
assert gpu_gb >= 70, f'需要 A100 80GB；当前只有 {gpu_gb:.1f}GB'
assert kernel['causal_conv1d'] and kernel['flash_linear_attention'], (
    '0B kernels 未在当前进程生效；重启 session 后从第 1 格再跑。'
)
torch.set_float32_matmul_precision('high')

bundle = b1.load_training_data(ROOT, TRAIN_PATH)
saved_folds = np.load(FOLD_SOURCE, allow_pickle=True)
assert saved_folds['row_ids'].astype(str).tolist() == bundle.row_ids.astype(str).tolist()
folds = saved_folds['folds'].astype(int)
assert sorted(np.unique(folds).tolist()) == [0, 1, 2]
print({'rows': len(bundle.texts), 'fold_sizes': np.bincount(folds).tolist(), 'users': len(np.unique(bundle.user_ids))})

for fold in range(3):
    paths = b15.locate_task1_fold(ROOT, fold)
    print(f'fold {fold}:', {key: str(value) for key, value in paths.items() if key != 'base'})


## CoT 结论（防止混淆）

| 实验 | 可否下结论 | Fold-0 结果 |
|---|---:|---:|
| silent pairwise/ranking 改判 | 可以 | Macro-F1 `−0.0057 ~ −0.0128`，失败 |
| visible CoT generation | 不可以 | 48/48 到 96 tokens 截断，0 个合法 `FINAL` |

所以“CoT 已经证明无用”不准确；准确说法是：**生成 harness 失败，排序改判失败。** B15 检验的是无需生成文字的 latent reasoning/readout 假设。


In [ ]:
#@title 3. Fold 0 配置（预注册，不要看结果后改 alpha）
FOLD0_CFG = b15.LatentReadoutConfig(
    fold=0,
    selected_layers=(15, 31, 47, 63),
    c_grid=(0.001, 0.01, 0.1),
    primary_blend_alpha=0.25,
    diagnostic_blend_alphas=(0.50,),
    extraction_batch_size=2,
    extraction_chunk_size=128,
    gate_weighted_f1_delta=0.008,
    gate_macro_f1_delta=-0.003,
    gate_behavior_recall_delta=-0.03,
    gate_attempt_recall_delta=-0.03,
)
print(FOLD0_CFG)
print('Hidden cache upper bound per fold ≈', round(len(bundle.texts) * 4 * 4 * 5120 * 2 / 2**30, 2), 'GiB before compression')


In [ ]:
#@title 4. 跑 / 恢复 Fold 0 latent extraction + linear readout
fold0_decision = None
if RUN_FOLD0:
    started = time.perf_counter()
    fold0_decision = b15.run_latent_readout_fold(
        bundle=bundle,
        folds=folds,
        root=ROOT,
        output_dir=OUT / 'fold_0',
        config=FOLD0_CFG,
    )
    print('elapsed min:', round((time.perf_counter() - started) / 60, 1))
else:
    path = OUT / 'fold_0/B15_FOLD_DECISION.json'
    if path.exists(): fold0_decision = json.loads(path.read_text(encoding='utf-8'))

if fold0_decision:
    display(pd.read_csv(OUT / 'fold_0/B15_FOLD_SUMMARY.csv'))
    display(pd.read_csv(OUT / 'fold_0/B15_PROBE_SELECTION.csv').head(12))
    print(json.dumps(fold0_decision, indent=2))


In [ ]:
#@title 5. Fold 0 gate（只认固定 25% blend）
if fold0_decision is None:
    raise RuntimeError('先完成第 4 格')

gate = bool(fold0_decision['passed'])
print({
    'PASS': gate,
    'primary': fold0_decision['primary'],
    'chosen_layer': fold0_decision['chosen_layer'],
    'chosen_c': fold0_decision['chosen_c'],
    'deltas': fold0_decision['deltas'],
    'bootstrap': fold0_decision['paired_bootstrap'],
})
if gate:
    print('PASS：把 RUN_CONFIRMATION=True，继续第 6 格；layer/C/alpha 已冻结。')
else:
    print('FAIL：停止 B15，不跑另外两折。可见 CoT 仍未被严格验证，但不值得用 27B 生成烧算力。')


In [ ]:
#@title 6. PASS 后冻结 layer/C，跑 / 恢复 Fold 1、2
confirmation = []
if RUN_CONFIRMATION:
    if not fold0_decision['passed']:
        raise RuntimeError('Fold 0 gate FAIL；禁止跑 confirmation')
    frozen_layer = int(fold0_decision['chosen_layer'])
    frozen_c = float(fold0_decision['chosen_c'])
    for fold in (1, 2):
        cfg = b15.LatentReadoutConfig(
            fold=fold,
            selected_layers=(15, 31, 47, 63),
            fixed_layer=frozen_layer,
            fixed_c=frozen_c,
            primary_blend_alpha=0.25,
            diagnostic_blend_alphas=(0.50,),
            extraction_batch_size=2,
            extraction_chunk_size=128,
        )
        result = b15.run_latent_readout_fold(
            bundle=bundle,
            folds=folds,
            root=ROOT,
            output_dir=OUT / f'fold_{fold}',
            config=cfg,
        )
        confirmation.append(result)
        display(pd.read_csv(OUT / f'fold_{fold}/B15_FOLD_SUMMARY.csv'))
        print({'fold': fold, 'passed': result['passed'], 'deltas': result['deltas']})
else:
    print('RUN_CONFIRMATION=False；Fold 0 PASS 后才打开。')


In [ ]:
#@title 7. 汇总与下载轻量报告（不打包 hidden cache）
import shutil

completed = b15.summarize_completed_folds(
    [OUT / f'fold_{fold}' for fold in range(3)], OUT / 'SUMMARY'
)
print(json.dumps(completed, indent=2))

package = Path('/content/B15_LATENT_READOUT_REPORT')
if package.exists(): shutil.rmtree(package)
package.mkdir(parents=True)
for fold in range(3):
    source = OUT / f'fold_{fold}'
    if not source.exists(): continue
    target = package / f'fold_{fold}'
    target.mkdir()
    for name in (
        'B15_FOLD_DECISION.json', 'B15_FOLD_SUMMARY.csv',
        'B15_PROBE_SELECTION.csv', 'B15_FOLD_PREDICTIONS.csv',
        'B15_FOLD_OUTPUTS.npz', 'B15_PROMPT_AUDIT.csv',
    ):
        if (source / name).exists(): shutil.copy2(source / name, target / name)
for path in (OUT / 'SUMMARY').glob('*'):
    shutil.copy2(path, package / path.name)
archive = shutil.make_archive('/content/B15_LATENT_READOUT_REPORT', 'zip', package)
print('Report:', archive)
files.download(archive)
